# Compute Trust Scores

This notebook implements the automated update of the trust model for a Unity Catalog schema.  
When you run this notebook, all attributes in the trust model are updated in the table `<CATALOG_NAME>.trustmodel.trust_scores`, and the trust score for all tables is stored in the tag **'Trust Score'**.


In [0]:
CATALOG_NAME = 'workspace'
SCHEMA_NAME = 'adventureworks'

##Helper functions

In [0]:
# check if a table has a description 
def hasTableComment(catalogName, schemaName, tableName):
    query = f"SELECT IFF(comment is null, false, true) as tableComment FROM {catalogName}.information_schema.tables WHERE table_schema = '{schemaName}' AND table_name = '{tableName}'"
    tableComment = spark.sql(query)  
    return tableComment.first()["tableComment"]

# check if a table has a markdown description
def hasMarkdownDescription(catalogName, schemaName, tableName):
    query = f"""
        SELECT comment 
        FROM {catalogName}.information_schema.tables
        WHERE table_schema = '{schemaName}'
          AND table_name = '{tableName}'
    """
    result = spark.sql(query).first()
    if result is None or result["comment"] is None:
        return False

    comment = result["comment"]

    # simpele regex checks voor Markdown
    import re
    md_patterns = [
        r"\*\*.+?\*\*",   # bold
        r"^#+\s",         # heading
        r"\n\s*[-*]\s",   # unordered list
        r"\n[0-9]+\.\s",  # ordered list
        r"`[^`]+`",       # inline code
        r"^>",            # blockquote
    ]

    for pat in md_patterns:
        if re.search(pat, comment, flags=re.MULTILINE):
            return True
    return False
    
# check if all columns in a table have a description 
def hasColumnComment(catalogName, schemaName, tableName):
    query = f"SELECT EVERY(comment is not null) as columnComment FROM {catalogName}.information_schema.columns WHERE table_schema = '{schemaName}' AND table_name = '{tableName}'"
    columnComment = spark.sql(query)   
    return columnComment.first()["columnComment"]

# check if a table description contains an "Owner: ..." line
# True when the table comment contains a line starting with "Owner: <readable text>" (Markdown like **Owner:** and `value`)
def hasOwner(catalogName, schemaName, tableName):
    query = f"""
        SELECT comment
        FROM {catalogName}.information_schema.tables
        WHERE table_schema = '{schemaName}'
          AND table_name   = '{tableName}'
    """
    row = spark.sql(query).first()
    if not row or row["comment"] is None:
        return False

    comment = row["comment"]

    import re

    # 1) Normalise frequently used Markdown surrounding label and value
    txt = comment
    # **Owner:** or __Owner:__  -> Owner:
    txt = re.sub(r"(?i)(?:\*\*|__)\s*Owner\s*:\s*(?:\*\*|__)", "Owner: ", txt)
    # **Owner**: or __Owner__:  -> Owner:
    txt = re.sub(r"(?i)(?:\*\*|__)\s*Owner\s*(?:\*\*|__)\s*:\s*", "Owner: ", txt)
    # *Owner:* / _Owner:_ / *Owner*: / _Owner_: -> Owner:
    txt = re.sub(r"(?i)(?:\*|_)\s*Owner\s*:\s*(?:\*|_)", "Owner: ", txt)
    txt = re.sub(r"(?i)(?:\*|_)\s*Owner\s*(?:\*|_)\s*:\s*", "Owner: ", txt)
    # remove inline-code backticks surrounding the value: `sales_team` -> sales_team
    txt = re.sub(r"`([^`]+)`", r"\1", txt)

    # 2) Find a line starting with Owner: and take the value till end of line
    m = re.search(r"(?im)^\s*Owner\s*:\s*(.+?)\s*$", txt)
    if not m:
        return False

    value = m.group(1).strip()

    # 3) "Readable": at least one alphanumerical character in the value
    return len(re.sub(r"[^A-Za-z0-9]+", "", value)) >= 1


# check how many weeks a table is in production
def getWeeksInProduction(catalogName, schemaName, tableName):
    query = f"SELECT FLOOR(DATEDIFF(GETDATE(), created)/7) as weeksInProduction FROM {catalogName}.information_schema.tables WHERE table_schema = '{schemaName}' AND table_name = '{tableName}'"
    weeksInProduction = spark.sql(query)   
    return weeksInProduction.first()["weeksInProduction"]

# Freshness: in elk geval kan je last altered date uit het informatie schema halen. Uitgebreidere checks zijn mogelijk afhankelijk van de freshness implementatie.  
# Has data checks: in elk geval kan je constraints op de tabel checken. Meer uitgebreide DQC's hangt af van de implementatie
# Access specificity: ik weet niet precies wat ze hier willen, maar je kan iig zien wie er toegang tot tabellen heeft, dus er moet wel iets mee te doen zijn. 
# Cost budget tags: Tags zijn eenvoudig te querien via het information schema. Moet je alleen wel iets van cost budgets in tags implementeren natuurlijk.

##Determine and persist score details for a given table

In [0]:
def determineTrustScoreDetails(catalogName, schemaName, tableName):
    hasComments = hasTableComment(catalogName, schemaName, tableName)
    hasMDDescription = hasMarkdownDescription(catalogName, schemaName, tableName)
    allColumnsHaveComments = hasColumnComment(catalogName, schemaName, tableName)
    hasHumanOwner = hasOwner(catalogName, schemaName, tableName)
    weeksInProduction = getWeeksInProduction(catalogName, schemaName, tableName)

    upsert_query = f"""
    MERGE INTO {catalogName}.trustmodel.trust_scores AS target
    USING (
        SELECT 
            '{catalogName}' AS catalogName, 
            '{schemaName}' AS schemaName, 
            '{tableName}' AS tableName, 
            '{hasComments}' AS hasComments, 
            '{hasMDDescription}' AS hasMarkdownDescription,
            '{allColumnsHaveComments}' AS allColumnsHaveComments, 
            '{hasHumanOwner}' AS hasHumanOwner, 
            '{weeksInProduction}' AS weeksInProduction
    ) AS source
    ON target.catalogName = source.catalogName 
       AND target.schemaName = source.schemaName 
       AND target.tableName = source.tableName
    WHEN MATCHED THEN
        UPDATE SET 
            target.hasComments = source.hasComments, 
            target.hasMarkdownDescription = source.hasMarkdownDescription,
            target.allColumnsHaveComments = source.allColumnsHaveComments, 
            target.hasHumanOwner = source.hasHumanOwner, 
            target.weeksInProduction = source.weeksInProduction
    WHEN NOT MATCHED THEN
        INSERT (
            catalogName, 
            schemaName, 
            tableName, 
            hasComments,
            hasMarkdownDescription, 
            allColumnsHaveComments, 
            hasHumanOwner, 
            weeksInProduction
        )
        VALUES (
            source.catalogName, 
            source.schemaName, 
            source.tableName, 
            source.hasComments, 
            source.hasMarkdownDescription,
            source.allColumnsHaveComments, 
            source.hasHumanOwner, 
            source.weeksInProduction
        )
    """
    spark.sql(upsert_query)

##Determine and persist the trust score details for all tables in CATALOG_NAME.SCHEMA_NAME.

In [0]:
tables_query = f"SELECT table_name FROM {CATALOG_NAME}.information_schema.tables WHERE table_schema = '{SCHEMA_NAME}'"
tables_df = spark.sql(tables_query)

for row in tables_df.collect():
    table_name = row["table_name"]
    determineTrustScoreDetails(CATALOG_NAME, SCHEMA_NAME, table_name)

##Determine and persist the trust score for all records.

In [0]:
update_query = """
UPDATE workspace.trustmodel.trust_scores
SET trust_score = 15 * CAST(hasComments AS INT) + 20 * CAST(hasMarkdownDescription AS INT) + 15 * CAST(allColumnsHaveComments AS INT) + 16 * CAST(hasHumanOwner AS INT) + 8 * LEAST(weeksInProduction, 5)
"""
spark.sql(update_query)

For debugging purposes, query the results.

In [0]:
%sql
SELECT * FROM workspace.trustmodel.trust_scores

##Persist the trust scores in the tag 'Trust Score'.

In [0]:

# Read the driving table
trust_scores = spark.table("workspace.trustmodel.trust_scores").collect()

for row in trust_scores:
    catalog = row["catalogName"]
    schema  = row["schemaName"]
    table   = row["tableName"]
    score   = row["trust_score"]

    fq_table = f"{catalog}.{schema}.{table}"
    try:
        spark.sql(f"""
            ALTER TABLE {fq_table}
            SET TAGS ('Trust Score' = '{score}')
        """)
    except Exception as e:
        print(f"❌ Error in {fq_table}: {e}")
